In [ ]:
import pandas as pd
import searoute as sr

In [ ]:
from _helpers import mock_snakemake
snakemake = mock_snakemake(
    "prepare_chokepoints",
)

### File paths

In [ ]:
trade_options_fn = snakemake.input.trade_options
bus_locations_fn = snakemake.input.bus_locations

# Output
trade_options_chokepoints_fn = snakemake.output.trade_options_chokepoints

### Load data

In [ ]:
trade_options = pd.read_csv(trade_options_fn, header=0)
bus_locations = pd.read_csv(bus_locations_fn, header=0)

print(f"Trade options: {len(trade_options)} routes")
print(f"Bus locations: {len(bus_locations)} regions")

# Build region -> (lon, lat) mapping for searoute
region_coords = {
    row["region_name"]: [row["long"], row["lat"]]
    for _, row in bus_locations.iterrows()
}

print(f"\nRegion coordinates:")
for name, coords in region_coords.items():
    print(f"  {name}: {coords}")

### Compute chokepoints per route using `searoute`

For each shipping route in `trade_opt.csv`, compute the shortest sea route and record which passages (chokepoints) are traversed.

In [ ]:
# Chokepoints to consider from config (mga-chokepoints threshold_value)
config_chokepoints = set(
    snakemake.config.get("scenario", {})
    .get("mga-chokepoints", {})
    .get("modifiers", {})
    .get("mga", {})
    .get("threshold_value", [])
)
print(f"Chokepoints to track (from config): {sorted(config_chokepoints)}")

chokepoints_list = []

for idx, row in trade_options.iterrows():
    r_from = row["region_from"]
    r_to = row["region_to"]

    # Only compute for shipping routes
    if row["shipping"] != 1:
        chokepoints_list.append("")
        continue

    origin = region_coords.get(r_from)
    destination = region_coords.get(r_to)

    if origin is None or destination is None:
        print(f"  Warning: missing coordinates for {r_from} or {r_to}")
        chokepoints_list.append("")
        continue

    try:
        route = sr.searoute(
            origin, destination,
            restrictions=[],  # allow all passages so we can detect them
            return_passages=True,
        )
        passages = route["properties"].get("traversed_passages", [])
        # Filter to only chokepoints listed in config
        relevant = [p for p in passages if p in config_chokepoints]
        chokepoints_list.append(";".join(relevant) if relevant else "")
    except Exception as e:
        print(f"  Error computing route {r_from} -> {r_to}: {e}")
        chokepoints_list.append("")

trade_options["chokepoints"] = chokepoints_list

print(f"\nRoutes with config chokepoints: {(trade_options['chokepoints'] != '').sum()} / {len(trade_options)}")

In [ ]:
# Display routes and their chokepoints
display_cols = ["region_from", "region_to", "chokepoints"]
routes_with_cp = trade_options[trade_options["chokepoints"] != ""][display_cols]
print(f"Routes passing through chokepoints ({len(routes_with_cp)}):")
routes_with_cp

### Save enriched trade options

In [ ]:
trade_options.to_csv(trade_options_chokepoints_fn, index=False)
print(f"Saved to {trade_options_chokepoints_fn}")

### Map: Shipping routes and chokepoints

Global map of all shipping routes, colored by whether they pass through chokepoints. Chokepoint locations are marked with red stars.

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np

# Known chokepoint coordinates (lon, lat) — only for config chokepoints
all_chokepoint_coords = {
    "babalmandab": [43.3, 12.6],
    "bosporus":    [29.0, 41.1],
    "gibraltar":   [-5.6, 36.0],
    "suez":        [32.3, 30.5],
    "panama":      [-79.5, 9.1],
    "ormuz":       [56.3, 26.6],
    "northwest":   [-95.0, 74.0],
    "malacca":     [101.0, 2.5],
    "sunda":       [105.8, -6.1],
    "chili":       [-75.0, -55.0],
    "south_africa":[ 20.0, -34.8],
    "bering":      [-169.0, 65.5],
}
chokepoint_coords = {k: v for k, v in all_chokepoint_coords.items() if k in config_chokepoints}

print(f"Plotting {len(chokepoint_coords)} config chokepoints: {sorted(chokepoint_coords.keys())}")

# Compute sea routes for all shipping pairs (store geometries for plotting)
routes_geo = []
for idx, row in trade_options.iterrows():
    if row["shipping"] != 1:
        continue
    r_from = row["region_from"]
    r_to = row["region_to"]
    origin = region_coords.get(r_from)
    destination = region_coords.get(r_to)
    if origin is None or destination is None:
        continue
    try:
        route = sr.searoute(origin, destination, restrictions=[], return_passages=True)
        cp_str = row.get("chokepoints", "")
        has_cp = pd.notna(cp_str) and cp_str != ""
        routes_geo.append({
            "coords": route["geometry"]["coordinates"],
            "has_chokepoint": has_cp,
            "chokepoints": cp_str if has_cp else "",
            "from": r_from,
            "to": r_to,
        })
    except Exception:
        pass

print(f"Computed {len(routes_geo)} route geometries")

In [ ]:
fig = plt.figure(figsize=(10, 7))
ax = plt.axes(projection=ccrs.Robinson())

# Map features
ax.add_feature(cfeature.LAND, facecolor="#f0f0f0", edgecolor="none")
ax.add_feature(cfeature.OCEAN, facecolor="#dbe9f4")
ax.add_feature(cfeature.COASTLINE, linewidth=0.3, color="grey")
ax.add_feature(cfeature.BORDERS, linewidth=0.2, color="lightgrey")
ax.set_global()

proj = ccrs.Robinson()

# Plot routes without chokepoints first (background, grey)
for r in routes_geo:
    if not r["has_chokepoint"]:
        coords = np.array(r["coords"])
        ax.plot(coords[:, 0], coords[:, 1],
                color="#cccccc", linewidth=0.4, alpha=0.5,
                transform=ccrs.Geodetic())

# Plot routes with chokepoints on top (colored)
for r in routes_geo:
    if r["has_chokepoint"]:
        coords = np.array(r["coords"])
        ax.plot(coords[:, 0], coords[:, 1],
                color="#d62728", linewidth=0.6, alpha=0.35,
                transform=ccrs.Geodetic())

# Plot region locations
for name, (lon, lat) in region_coords.items():
    ax.plot(lon, lat, marker="o", color="#1f77b4", markersize=5,
            transform=ccrs.PlateCarree(), zorder=10)
    ax.text(lon + 2, lat + 2, name.replace("_", " "), fontsize=5,
            transform=ccrs.PlateCarree(), zorder=10,
            ha="left", va="bottom", color="#333333")

# Plot chokepoint locations
for cp_name, (lon, lat) in chokepoint_coords.items():
    ax.plot(lon, lat, marker="*", color="#d62728", markersize=12,
            markeredgecolor="white", markeredgewidth=0.5,
            transform=ccrs.PlateCarree(), zorder=20)
    # Label with nicer formatting
    label = cp_name.replace("_", " ").title()
    ax.text(lon + 3, lat - 3, label, fontsize=6, fontweight="bold",
            transform=ccrs.PlateCarree(), zorder=20,
            ha="left", va="top", color="#d62728",
            bbox=dict(boxstyle="round,pad=0.15", facecolor="white", alpha=0.7, edgecolor="none"))

# Legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color="#d62728", linewidth=1.5, alpha=0.6, label="Route through chokepoint"),
    Line2D([0], [0], color="#cccccc", linewidth=1.5, alpha=0.8, label="Route without chokepoint"),
    Line2D([0], [0], marker="*", color="#d62728", linestyle="None",
           markersize=12, markeredgecolor="white", label="Chokepoint"),
    Line2D([0], [0], marker="o", color="#1f77b4", linestyle="None",
           markersize=6, label="Region"),
]
ax.legend(handles=legend_elements, loc="lower left", fontsize=8, framealpha=0.9)

ax.set_title("Global Shipping Routes and Maritime Chokepoints")

plt.tight_layout()
plt.show()
print(f"\nRoutes through chokepoints: {sum(1 for r in routes_geo if r['has_chokepoint'])} / {len(routes_geo)}")